# CellIQ: training notebook (lung)

This notebook teaches an AI to sort lung images into groups, checks how well it works, and draws a **Grad-CAM heatmap** showing where it looked.

**How to use it (in Google Colab):**
1. Menu: Runtime > Change runtime type > pick a **GPU**.
2. Put your dataset zip on Google Drive (download it from Kaggle first).
3. Change the settings in **Step 1** only, then run the cells from top to bottom.

> Research and educational prototype. Not for real diagnosis.

**The 6 ideas to explain to the judges:** load data, preprocess, load a pretrained model, train, evaluate, explain with a heatmap.

## Step 1: Settings (the only cell you need to edit)

In [ ]:
# Pick ONE: "lung_slides" (LC25000 microscope slides) or "lung_ct" (IQ-OTH/NCCD CT scans)
DATASET = "lung_slides"

# Path to the folder that directly contains the class folders (one folder per group).
# LC25000 example:   /content/data/lung_colon_image_set/lung_image_sets
# IQ-OTH/NCCD example: /content/data/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset
DATA_DIR = "/content/data/lung_colon_image_set/lung_image_sets"

# Optional: unzip a dataset from Google Drive. Set to "" to skip.
ZIP_PATH = ""            # e.g. "/content/drive/MyDrive/lung-and-colon-cancer-histopathological-images.zip"
UNZIP_TO = "/content/data"

IMG_SIZE = 224           # every image is resized to 224 x 224
BATCH_SIZE = 32
HEAD_EPOCHS = 3          # phase 1: train only the new last layer
FINETUNE_EPOCHS = 3      # phase 2: gently adjust the whole network
SEED = 42
OUT_DIR = "/content/cellIQ_" + DATASET

## Step 2: Tools and optional unzip

In [ ]:
import os, json, random, zipfile, re
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device, "(GPU is much faster; use Runtime > Change runtime type)")

if ZIP_PATH:
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(UNZIP_TO)
    print("Unzipped to", UNZIP_TO)
os.makedirs(OUT_DIR, exist_ok=True)

## Step 3: Load the data and split it

Each folder name is a label. We split the images into **train** (the AI learns from these), **validation** (we tune with these) and **test** (used once at the end for an honest score).

**Important honesty point:**
- LC25000 contains augmented copies of about 750 original photos, so a random split can put near-copies in both train and test. The score will look better than it really is. Say this to the judges.
- IQ-OTH/NCCD has several slices per patient, but the files often have no patient ID. For CT we split in blocks of neighbouring files, which approximates "keep a patient together". It is an approximation, so say so.

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
class_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
print("Groups found:", class_dirs)
assert len(class_dirs) >= 2, "DATA_DIR should contain one folder per group. Check the path."
class_names = class_dirs
files, labels = [], []
for idx, d in enumerate(class_dirs):
    folder = os.path.join(DATA_DIR, d)
    names = sorted(os.listdir(folder), key=lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)])
    for f in names:
        if f.lower().endswith(IMG_EXT):
            files.append(os.path.join(folder, f)); labels.append(idx)
files = np.array(files); labels = np.array(labels)
print("Total images:", len(files), "| per group:", np.bincount(labels))

def block_split(files, labels, block=10):
    """CT only: keep runs of neighbouring files together (approximate patient grouping)."""
    train, val, test = [], [], []
    rng = random.Random(SEED)
    for c in np.unique(labels):
        idxs = np.where(labels == c)[0]
        blocks = [idxs[i:i+block] for i in range(0, len(idxs), block)]
        rng.shuffle(blocks)
        n = len(blocks); a, b = int(n*0.7), int(n*0.85)
        for bl in blocks[:a]: train += list(bl)
        for bl in blocks[a:b]: val += list(bl)
        for bl in blocks[b:]: test += list(bl)
    return np.array(train), np.array(val), np.array(test)

if DATASET == "lung_ct":
    tr, va, te = block_split(files, labels)
else:
    idx = np.arange(len(files))
    tr, tmp = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.50, stratify=labels[tmp], random_state=SEED)

print("train / val / test sizes:", len(tr), len(va), len(te))

## Step 4: Preprocess (get images ready for the AI)

Resize, turn into numbers, and normalise with the values the pretrained network expects. For the **training** images only, we add random flips, small rotations and brightness changes (augmentation) so the AI doesn't just memorise.

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean, std)])

class ScanDataset(Dataset):
    def __init__(self, idxs, tf): self.idxs, self.tf = idxs, tf
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        img = Image.open(files[j]).convert("RGB")
        return self.tf(img), int(labels[j])

train_dl = DataLoader(ScanDataset(tr, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_dl   = DataLoader(ScanDataset(va, eval_tf),  batch_size=BATCH_SIZE, num_workers=2)
test_dl  = DataLoader(ScanDataset(te, eval_tf),  batch_size=BATCH_SIZE, num_workers=2)

## Step 5: The model (transfer learning)

We start from **EfficientNet-B0**, already trained on millions of everyday photos, so it can already see edges and shapes. We replace its last layer with one that outputs our groups. That is called *transfer learning*.

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

# Count images per group and give rarer groups a bigger weight, so the AI does not ignore them
counts = np.bincount(labels[tr], minlength=len(class_names))
weights = torch.tensor(counts.sum() / (len(class_names) * counts), dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
print("Class weights:", weights.cpu().numpy().round(2))

## Step 6: Train

Phase 1 trains only the new last layer (fast). Phase 2 lets the whole network adjust very gently (small learning rate). We keep the version that did best on the validation set.

In [ ]:
def run_epoch(dl, train, opt=None):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y)
            correct += (out.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

best_val, best_state = 1e9, None
def fit(epochs, lr, tag):
    global best_val, best_state
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    for ep in range(epochs):
        tl, ta = run_epoch(train_dl, True, opt)
        vl, va_ = run_epoch(val_dl, False)
        print(f"{tag} epoch {ep+1}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va_:.3f}")
        if vl < best_val:
            best_val = vl; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# Phase 1: freeze everything except the new last layer
for p in model.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
fit(HEAD_EPOCHS, 1e-3, "Phase 1 (head only)")

# Phase 2: unfreeze all, tiny learning rate
for p in model.parameters(): p.requires_grad = True
fit(FINETUNE_EPOCHS, 1e-4, "Phase 2 (fine-tune)")

model.load_state_dict(best_state)
torch.save(best_state, os.path.join(OUT_DIR, "model.pt"))
json.dump({"classes": class_names, "img_size": IMG_SIZE, "dataset": DATASET},
          open(os.path.join(OUT_DIR, "meta.json"), "w"))
print("Saved best model to", OUT_DIR)

## Step 7: Evaluate honestly (use the test set once)

Accuracy alone can mislead. Look at **sensitivity** (of the real cancers, how many did we catch?) and **specificity** (of the healthy ones, how many did we correctly clear?), plus the confusion matrix and AUC.

In [ ]:
model.eval()
all_prob, all_true = [], []
with torch.no_grad():
    for x, y in test_dl:
        p = F.softmax(model(x.to(device)), dim=1).cpu().numpy()
        all_prob.append(p); all_true.append(y.numpy())
prob = np.concatenate(all_prob); true = np.concatenate(all_true); pred = prob.argmax(1)

print(classification_report(true, pred, target_names=class_names, digits=3))
cm = confusion_matrix(true, pred)
print("Confusion matrix (rows = truth, columns = AI guess):\n", cm)

for i, name in enumerate(class_names):
    tp = cm[i, i]; fn = cm[i].sum() - tp
    fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
    print(f"{name}: sensitivity {tp/max(tp+fn,1):.3f} | specificity {tn/max(tn+fp,1):.3f}")

try:
    print("AUC (one-vs-rest, average):", round(roc_auc_score(true, prob, multi_class="ovr"), 3))
except Exception as e:
    print("AUC could not be computed:", e)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=30, ha="right"); ax.set_yticklabels(class_names)
for i in range(len(cm)):
    for j in range(len(cm)): ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("AI guess"); ax.set_ylabel("Truth"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

## Step 8: Grad-CAM heatmap (where did the AI look?)

Grad-CAM asks: which parts of the image, when changed, would most change the answer? Those parts are drawn in red. It shows where the AI looked, **not** proof that it is right.

In [ ]:
target_layer = model.features[-1]
acts, grads = {}, {}
target_layer.register_forward_hook(lambda m, i, o: acts.__setitem__("v", o))
target_layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__("v", go[0]))

def gradcam(path):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval(); model.zero_grad()
    out = model(x); cls = out.argmax(1).item()
    out[0, cls].backward()
    w = grads["v"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * acts["v"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    prob = F.softmax(out, dim=1)[0].detach().cpu().numpy()
    return img.resize((IMG_SIZE, IMG_SIZE)), cam.detach().cpu().numpy(), cls, prob

# Show 4 random test images
for j in random.sample(list(te), 4):
    img, cam, cls, prob = gradcam(files[j])
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
    ax[0].imshow(img); ax[0].set_title("Truth: " + class_names[labels[j]]); ax[0].axis("off")
    ax[1].imshow(img); ax[1].imshow(cam, cmap="jet", alpha=0.45)
    ax[1].set_title(f"AI: {class_names[cls]} ({prob[cls]*100:.0f}%)"); ax[1].axis("off")
    plt.show()

## Step 9: Failure gallery (judges like honesty)

Look at images the AI got **wrong** and ask why. Put one or two in your presentation.

In [ ]:
wrong = [te[i] for i in np.where(pred != true)[0]]
print("Number of wrong test predictions:", len(wrong))
for j in wrong[:4]:
    img, cam, cls, prob = gradcam(files[j])
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
    ax[0].imshow(img); ax[0].set_title("Truth: " + class_names[labels[j]]); ax[0].axis("off")
    ax[1].imshow(img); ax[1].imshow(cam, cmap="jet", alpha=0.45)
    ax[1].set_title(f"AI said: {class_names[cls]} ({prob[cls]*100:.0f}%)"); ax[1].axis("off")
    plt.show()

## What to say in the presentation

- **Data:** where it came from, how many images, what the groups mean, and its limits (see Step 3).
- **Model:** a pretrained network (EfficientNet-B0) adapted to our groups.
- **Training:** train / validation / test split; augmentation; class weights.
- **Results:** sensitivity, specificity, confusion matrix, AUC, plus a failure example.
- **Heatmap:** shows where the AI looked; not proof it is right.
- **Limits:** research prototype; a doctor makes the diagnosis.

Next step: save `model.pt` and `meta.json` and load them in the small API that the dashboard calls.